In [22]:
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt


RUNS = {
    0: [
        ("Well_3_1405", "data/0/WELL-00003_20170514010000.csv"),
        ("Well_7_0108", "data/0/WELL-00007_20170801180000.csv"),
        ("Well_5_1708", "data/0/WELL-00005_20170817020000.csv"),
        ("Well_6_2008", "data/0/WELL-00006_20170820010000.csv"),
    ],
    1: [
        ("Well_6_0108", "data/1/WELL-00006_20170801063614.csv"),
        ("Well_6_0208", "data/1/WELL-00006_20170802123000.csv"),
        ("Sim_0112", "data/1/SIMULATED_00112.csv"),
        ("Sim_0044", "data/1/SIMULATED_00044.csv"),
    ],
    2: [
        ("Well_10_1812", "data/2/WELL-00010_20171218200131.csv"),
        ("Well_3_0602", "data/2/WELL-00003_20180206182917.csv"),
        ("Well_12_2003", "data/2/WELL-00012_20170320033022.csv"),
        ("Sim_02", "data/2/SIMULATED_00002.csv"),
    ],
    3: [
        ("Well_14_2810", "data/3/WELL-00014_20171028080000.csv"),
        ("Well_14_1709", "data/3/WELL-00014_20170917150000.csv"),
        ("Well_14_2509", "data/3/WELL-00014_20170925010031.csv"),
        ("Sim_67", "data/3/SIMULATED_00067.csv"),
    ],
    4: [
        ("Well_1_1603", "data/4/WELL-00001_20170316120203.csv"),
        ("Well_7_1705", "data/4/WELL-00007_20170517210012.csv"),
        ("Well_10_1604", "data/4/WELL-00010_20180416080150.csv"),
        ("Well_14_1509", "data/4/WELL-00014_20170915090021.csv"),
    ],
    5: [
        ("Sim_17", "data/5/SIMULATED_00017.csv"),
        ("Well_15_1310", "data/5/WELL-00015_20171013140047.csv"),
        ("Sim_193", "data/5/SIMULATED_00193.csv"),
        ("Well_17_1403", "data/5/WELL-00017_20140314180000.csv"),
    ],
    6: [
        ("Well_2_1202", "data/6/WELL-00002_20140212170333.csv"),
        ("Sim_29", "data/6/SIMULATED_00029.csv"),
        ("Sim_207", "data/6/SIMULATED_00207.csv"),
        ("Sim_111", "data/6/SIMULATED_00111.csv"),
    ],
    7: [
        ("Drawn_10", "data/7/DRAWN_00010.csv"),
        ("Drawn_7", "data/7/DRAWN_00007.csv"),
        ("Well_6_1806", "data/7/WELL-00006_20180618110721.csv"),
        ("Well_1_2602", "data/7/WELL-00001_20170226220309.csv"),
    ],
    8: [
        ("Well_21_0905", "data/8/WELL-00021_20170509013517.csv"),
        ("Well_20_1004", "data/8/WELL-00020_20120410192326.csv"),
        ("Sim_00016", "data/8/SIMULATED_00016.csv"),
        ("Sim_00046", "data/8/SIMULATED_00046.csv"),
    ],
}

OUTPUT_ROOT = "../outputs"       
SHOW_PLOTS = False               

ISO_SCALER_PATH = "../../models/anomaly_scaler.joblib"
ISO_MODEL_PATH  = "../../models/anomaly_isolation_forest.joblib"
LSTM_PATH       = "../../models/lstm.pth"
LSTM_SCALER_PATH = "../../models/lstm_scaler.joblib"

ISO_FEATURES = ['rel_tot_dp_change', 'rel_ts_dp_change',
                'rel_p_ts_dp_change', 'Relative_Ratio_Dev']
LSTM_FEATURES = ISO_FEATURES + ['anomaly_score']  
WINDOW = 60
# =====================================================================

if not SHOW_PLOTS:
    matplotlib.use("Agg")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class PlainLSTM(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=32, num_layers=1, num_classes=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :]))


# ---------- load models once ----------
iso_scaler = joblib.load(ISO_SCALER_PATH)
iso_model = joblib.load(ISO_MODEL_PATH)
lstm_scaler = joblib.load(LSTM_SCALER_PATH)
model = PlainLSTM(input_dim=len(LSTM_FEATURES), hidden_dim=32, num_layers=1,
                  num_classes=4, dropout=0.3).to(device)
model.load_state_dict(torch.load(LSTM_PATH, map_location=device))
model.eval()


# ---------- processing ----------
def load_and_featurize(path):
    df = pd.read_csv(path)
    df = df.drop(columns=['timestamp', 'T-TPT', 'T-JUS-CKP', 'T-JUS-CKGL',
                          'P-JUS-CKGL', 'QGL'], errors='ignore')

    tmp = df.rename(columns={'P-PDG': 'Pump outlet pressure (Mean)',
                             'P-TPT': 'TS inlet pressure (Mean)',
                             'P-MON-CKP': 'TS outlet pressure (Mean)'})

    if 'class' in tmp.columns:
        tmp['class'] = np.where(tmp['class'] > 100, (tmp['class'] - 100) / 2, tmp['class'])

    p_cols = ['Pump outlet pressure (Mean)', 'TS inlet pressure (Mean)', 'TS outlet pressure (Mean)']
    n_nan = int(tmp[p_cols].isna().sum().sum())
    if n_nan:
        print(f"   warning: {n_nan} NaN pressure values, filled with ffill/bfill")
        tmp[p_cols] = tmp[p_cols].ffill().bfill()

    tmp['Differential pressure (Mean)'] = tmp['TS inlet pressure (Mean)'] - tmp['TS outlet pressure (Mean)']
    tmp['Pump to inlet DP'] = tmp['Pump outlet pressure (Mean)'] - tmp['TS inlet pressure (Mean)']
    tmp['Total system DP'] = tmp['Pump outlet pressure (Mean)'] - tmp['TS outlet pressure (Mean)']

    base = tmp.iloc[:60]
    tmp['rel_tot_dp_change'] = tmp['Total system DP'] / base['Total system DP'].median()
    tmp['rel_ts_dp_change'] = tmp['Differential pressure (Mean)'] / base['Differential pressure (Mean)'].median()
    tmp['rel_p_ts_dp_change'] = tmp['Pump to inlet DP'] / base['Pump to inlet DP'].median()

    tmp['Pressure_Ratio'] = (tmp['TS inlet pressure (Mean)'] - tmp['TS outlet pressure (Mean)']) / \
                            (tmp['Pump outlet pressure (Mean)'] - tmp['TS inlet pressure (Mean)'] + 1e-6)
    base_ratio = tmp['Pressure_Ratio'].iloc[:60].median()
    tmp['Relative_Ratio_Dev'] = (tmp['Pressure_Ratio'] / (base_ratio + 1e-6) - 1).abs()
    return tmp


def add_anomaly_score(tmp):
    X = iso_scaler.transform(tmp[ISO_FEATURES])
    tmp['anomaly_score'] = iso_model.decision_function(X)
    return tmp


def predict_states(tmp):
    preds = [0] * (WINDOW - 1)
    with torch.no_grad():
        for t in range(WINDOW - 1, len(tmp)):
            w = tmp[LSTM_FEATURES].iloc[t - WINDOW + 1: t + 1].values
            x = torch.tensor(lstm_scaler.transform(w), dtype=torch.float32).unsqueeze(0).to(device)
            preds.append(model(x).argmax(dim=1).item())
    tmp['predicted_state'] = preds
    return tmp


# ---------- plotting ----------
def make_plot(df, col, title, ylabel, out_path, smooth_window=100):
    fig, ax = plt.subplots(figsize=(16, 6))
    x = df['elapsed_seconds']
    if smooth_window > 1:
        ax.plot(x, df[col], color = 'skyblue', alpha=0.4, label=col)
        ax.plot(x, df[col].rolling(window=smooth_window, min_periods=1).mean(),
                color='darkblue', linewidth=2, label='Smoothed Trend')
    else:
        ax.plot(x, df[col], color='darkblue', linewidth=2, label=col)
    
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Time (elapsed seconds)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


def save_all_plots(tmp, name, cls):
    out_dir = os.path.join(OUTPUT_ROOT, str(cls))
    os.makedirs(out_dir, exist_ok=True)
    make_plot(tmp, 'Relative_Ratio_Dev', f'{name} - Relative Ratio Deviation Over Time',
              'Deviation Magnitude', os.path.join(out_dir, f'{name}_relative_ratio_deviation.png'))
    make_plot(tmp, 'anomaly_score', f'{name} - Anomaly Score Over Time',
              'Anomaly Magnitude', os.path.join(out_dir, f'{name}_anomaly_score.png'))
    make_plot(tmp, 'predicted_state', f'{name} - predicted label',
              'label', os.path.join(out_dir, f'{name}_predicted_label.png'), smooth_window=1)
    make_plot(tmp, 'class', f'{name} - actual label',
              'label', os.path.join(out_dir, f'{name}_actual_label.png'), smooth_window=1)


# ---------- main loop ----------
summary = []
for cls, runs in RUNS.items():
    for name, path in runs:
        print(f"[class {cls}] {name}")
        try:
            tmp = load_and_featurize(path)
            tmp = add_anomaly_score(tmp)
            tmp = predict_states(tmp)
            tmp['elapsed_seconds'] = tmp.index.astype(int)
            save_all_plots(tmp, name, cls)

            shares = tmp['predicted_state'].value_counts(normalize=True)
            summary.append({
                'class_folder': cls, 'name': name, 'n_samples': len(tmp),
                **{f'pred_{k}_share': round(float(shares.get(k, 0)), 4) for k in range(4)},
                'mean_anomaly_score': round(float(tmp['anomaly_score'].mean()), 4),
                'min_anomaly_score': round(float(tmp['anomaly_score'].min()), 4),
            })
        except Exception as e:
            print(f"   FAILED: {e}")
            summary.append({'class_folder': cls, 'name': name, 'error': str(e)})

summary_df = pd.DataFrame(summary)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(summary_df)

[class 0] Well_3_1405
[class 0] Well_7_0108
[class 0] Well_5_1708
[class 0] Well_6_2008
[class 1] Well_6_0108
[class 1] Well_6_0208
[class 1] Sim_0112
[class 1] Sim_0044
[class 2] Well_10_1812
[class 2] Well_3_0602
[class 2] Well_12_2003
[class 2] Sim_02
[class 3] Well_14_2810
[class 3] Well_14_1709
[class 3] Well_14_2509
[class 3] Sim_67
[class 4] Well_1_1603
[class 4] Well_7_1705
[class 4] Well_10_1604
[class 4] Well_14_1509
[class 5] Sim_17
[class 5] Well_15_1310
[class 5] Sim_193
[class 5] Well_17_1403
[class 6] Well_2_1202
[class 6] Sim_29
[class 6] Sim_207
[class 6] Sim_111
[class 7] Drawn_10
[class 7] Drawn_7
[class 7] Well_6_1806
[class 7] Well_1_2602
[class 8] Well_21_0905
[class 8] Well_20_1004
[class 8] Sim_00016
[class 8] Sim_00046
    class_folder          name  n_samples  pred_0_share  pred_1_share  \
0              0   Well_3_1405      17932        1.0000        0.0000   
1              0   Well_7_0108       7185        1.0000        0.0000   
2              0   Well_5_1